<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

## **Optimising Marketing with Artificial Intelligence (OMAI)**

<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

### **Data Preprocessing - Machine Learning Recommendation Models**

---

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

OMAI_distilbert_round3 = pd.read_csv('OMAI - Results - DistilBERT (Round 3) (Part 2).csv')
OMAI_distilbert_round3 = OMAI_distilbert_round3.drop('Unnamed: 0', axis = 1)
OMAI_distilbert_round3.iloc[0:1]

In [ ]:
OMAI_distilbert_round3['description'] = OMAI_distilbert_round3['description'].str.lower()

OMAI_productcatdesc_check = OMAI_distilbert_round3[['productid', 'title', 
                                                    'categories', 'description']].drop_duplicates()
OMAI_productcatdesc_check[0:5]

In [ ]:
import re
pd.set_option('display.max_colwidth', None)

OMAI_distilbert_round3['description'] = OMAI_distilbert_round3['description'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
OMAI_productcatdesc_check = OMAI_distilbert_round3[['productid', 'title', 
                                                    'categories', 'description']].drop_duplicates()
OMAI_productcatdesc_check[0:5]

In [ ]:
import string
string.punctuation

def remove_punctuation(text):
    punctuationfree = "".join([i for i in text if i not in string.punctuation])
    return punctuationfree

OMAI_distilbert_round3['description'] = OMAI_distilbert_round3['description'].apply(lambda x:remove_punctuation(x))
OMAI_productcatdesc_check = OMAI_distilbert_round3[['productid', 'title', 
                                                    'categories', 'description']].drop_duplicates()
OMAI_productcatdesc_check[0:5]

In [ ]:
OMAI_consoleencoded = pd.get_dummies(OMAI_distilbert_round3['console'], prefix = 'console').astype(int)
OMAI_distilbert_round3 = pd.concat([OMAI_distilbert_round3, OMAI_consoleencoded], axis = 1)
OMAI_distilbert_round3[0:1]

In [ ]:
from sklearn.preprocessing import StandardScaler

OMAI_pricescaler = StandardScaler()
OMAI_distilbert_round3['price_normalised'] = OMAI_pricescaler.fit_transform(OMAI_distilbert_round3[['price']])
OMAI_distilbert_round3[0:1]

In [ ]:
OMAI_distilbert_round3 = OMAI_distilbert_round3.drop(['review', 'categories', 'time', 'summary', 'sentiment_vader_posR3', 
                                                      'sentiment_vader_negR3', 'sentiment_vader_neuR3', 'bert_pred_label13', 
                                                      'bert_pred_label13word', 'sentiment_vaderR3', 'sentiment_score_vader_R3', 
                                                      'sentiment_score_vader_labelR3'], axis = 1, errors = 'ignore')

In [ ]:
OMAI_distilbert_round3['average_score'] = OMAI_distilbert_round3.groupby('productid')['score'].transform('mean')
OMAI_distilbert_round3['average_bert_pred_prob_pos'] = OMAI_distilbert_round3.groupby('productid')['bert_pred_prob_pos13'].transform('mean')
OMAI_distilbert_round3['average_bert_pred_prob_nonpos'] = OMAI_distilbert_round3.groupby('productid')['bert_pred_prob_nonpos13'].transform('mean')
OMAI_distilbert_round3['average_sentiment_vader'] = OMAI_distilbert_round3.groupby('productid')['sentiment_vader_compoundR3'].transform('mean')
OMAI_distilbert_round3['review_counts'] = OMAI_distilbert_round3.groupby('productid')['productid'].transform('count')

In [ ]:
OMAI_distilbert_round3 = OMAI_distilbert_round3.drop(['combined_text', 'userreviewid'], axis = 1)

In [ ]:
OMAI_distilbert_round3 = OMAI_distilbert_round3.drop(['score', 'sentiment_scoreR3', 'sentiment_score_labelR3',
                                                      'sentiment_vader_compoundR3', 'bert_pred_prob_nonpos13', 'bert_pred_prob_pos13'],
                                                    axis = 1, errors = 'ignore')

In [ ]:
OMAI_distilbert_round3[OMAI_distilbert_round3['productid'] == 'B0002IQD3I'].drop(columns = ['description'])
OMAI_distilbert_round3[OMAI_distilbert_round3['productid'] == 'B0002IQD3I'].drop(columns = ['description'])
OMAI_distilbert_round3[OMAI_distilbert_round3['productid'] == 'B00005O0I2'].drop(columns = ['description'])
OMAI_distilbert_round3[OMAI_distilbert_round3['productid'] == 'B00005NUIW'].drop(columns = ['description'])
OMAI_distilbert_round3[OMAI_distilbert_round3['productid'] == 'B00008URUA'].drop(columns = ['description'])
OMAI_distilbert_round3[OMAI_distilbert_round3['productid'] == 'B0002I9RRM'].drop(columns = ['description'])
OMAI_distilbert_round3[OMAI_distilbert_round3['productid'] == 'B00005O0I2'].drop(columns = ['description'])

In [ ]:
OMAI_distilbert_round3 = OMAI_distilbert_round3.drop_duplicates(subset = 'productid')
OMAI_distilbert_round3 = OMAI_distilbert_round3.reset_index(drop = True)
print(OMAI_distilbert_round3[OMAI_distilbert_round3.review_counts > 3])
OMAI_distilbert_round3 = OMAI_distilbert_round3[OMAI_distilbert_round3.review_counts > 3]

In [ ]:
def create_average_sentiment(average_score):
    if average_score > 3.0:
        return 1 
    elif average_score <= 3.0:
        return 0
OMAI_distilbert_round3['average_sentiment_score'] = OMAI_distilbert_round3['average_score'].apply(create_average_sentiment)

In [ ]:
OMAI_sentiment_check = OMAI_distilbert_round3[['productid', 'average_score', 
                                               'average_sentiment_score']].drop_duplicates()
OMAI_sentiment_check.iloc[0:50]

In [ ]:
def create_average_sentiment_label(average_sentiment_score):
    if average_sentiment_score == 1.0:
        return 'Positive' 
    elif average_sentiment_score == 0.0:
        return 'Non-Positive'
OMAI_distilbert_round3['average_sentiment_label'] = OMAI_distilbert_round3['average_sentiment_score'].apply(create_average_sentiment_label)

In [ ]:
OMAI_sentiment_check = OMAI_distilbert_round3[['productid', 'average_score',
                                               'average_sentiment_score', 
                                               'average_sentiment_label']].drop_duplicates()
OMAI_sentiment_check.iloc[0:50]

In [ ]:
k = 10
prior = 3.0

OMAI_distilbert_round3['adjusted_average_score'] = OMAI_distilbert_round3.apply(lambda observation: 
        ((observation['average_score'] * observation['review_counts']) + (prior * k)) / (observation['review_counts'] + k)
        if observation['review_counts'] < 10 
        else observation['average_score'], axis = 1)

In [ ]:
def create_adjusted_average_sentiment(adjusted_average_score):
    if adjusted_average_score > 3.0:
        return 1 
    elif adjusted_average_score <= 3.0:
        return 0
OMAI_distilbert_round3['adjusted_average_sentiment_score'] = OMAI_distilbert_round3['adjusted_average_score'].apply(create_adjusted_average_sentiment)

In [ ]:
OMAI_sentiment_check = OMAI_distilbert_round3[['productid', 'average_score', 'adjusted_average_score', 
                                               'adjusted_average_sentiment_score']].drop_duplicates()
OMAI_sentiment_check.iloc[0:50]

In [ ]:
def create_adjusted_average_sentiment_label(adjusted_average_sentiment_score):
    if adjusted_average_sentiment_score == 1.0:
        return 'Positive' 
    elif adjusted_average_sentiment_score == 0.0:
        return 'Non-Positive'
OMAI_distilbert_round3['adjusted_average_sentiment_label'] = OMAI_distilbert_round3['adjusted_average_sentiment_score'].apply(create_adjusted_average_sentiment_label)

In [ ]:
OMAI_sentiment_check = OMAI_distilbert_round3[['productid', 'average_score', 'adjusted_average_score', 'review_counts',
                                               'average_sentiment_label', 'adjusted_average_sentiment_score', 
                                               'adjusted_average_sentiment_label']].drop_duplicates()

OMAI_sentiment_check = OMAI_sentiment_check[OMAI_sentiment_check['review_counts'] < 10]
OMAI_sentimentlabelchanges = OMAI_sentiment_check[OMAI_sentiment_check['average_sentiment_label'] != 
                                                  OMAI_sentiment_check['adjusted_average_sentiment_label']]
OMAI_sentimentlabelchanges[0:10]

In [ ]:
k = 10
prior = 0.5

OMAI_distilbert_round3['adjusted_average_bert_pred_prob_pos'] = OMAI_distilbert_round3.apply(lambda observation: 
        ((observation['average_bert_pred_prob_pos'] * observation['review_counts']) + 
         (prior * k)) / (observation['review_counts'] + k)
        if observation['review_counts'] < 10 
        else observation['average_bert_pred_prob_pos'], axis = 1)

In [ ]:
k = 10
prior = 0.5

OMAI_distilbert_round3['adjusted_average_bert_pred_prob_nonpos'] = OMAI_distilbert_round3.apply(lambda observation: 
        ((observation['average_bert_pred_prob_nonpos'] * observation['review_counts']) + 
         (prior * k)) / (observation['review_counts'] + k)
        if observation['review_counts'] < 10 
        else observation['average_bert_pred_prob_nonpos'], axis = 1)

In [ ]:
k = 10
prior = 0.5

OMAI_distilbert_round3['adjusted_average_sentiment_vader'] = OMAI_distilbert_round3.apply(lambda observation: 
        ((observation['average_sentiment_vader'] * observation['review_counts']) + 
         (prior * k)) / (observation['review_counts'] + k)
        if observation['review_counts'] < 10 
        else observation['average_sentiment_vader'], axis = 1)

In [ ]:
OMAI_distilbert_round3_ds = OMAI_distilbert_round3.describe()
OMAI_distilbert_round3_ds.to_csv('OMAI - Data - Exploratory Data Analysis (MLRMs) (Descriptive Statistics).csv')

In [ ]:
OMAI_mlrmeda = OMAI_distilbert_round3.copy(deep = True)
OMAI_mlrmeda.to_csv('OMAI - Data - Machine Learning Recommendation Models (MLRMs).csv')